# ECMWF AIFS-ENS — Download

**Model:** ECMWF AIFS Ensemble — AI/ML-based ensemble system  
**Type:** AI probabilistic — 51 members (1 control + 50 perturbed)  
**Cycles:** 00Z and 12Z  

## Verified available types

| Type | Description | Size/step |
|------|-------------|----------|
| `cf` | Control forecast (1 member) | ~3.5 MB |
| `pf` | All 50 perturbed members combined | ~174 MB — **not used** |
| `em` | Ensemble mean | ~1 MB |
| `es` | Ensemble spread (std dev) | ~1.4 MB |

We download `em` and `es` for compact mean/spread products.

## Parameter differences vs IFS ENS

- Wind: `10si` (speed scalar) only — `10u`/`10v` components not in `em`/`es`
- No `tp` (total precipitation) in `em`/`es`
- `2t` and `msl` available ✓

## Files produced

```
data/YYYY-MM-DD_HHZ/
    aifs_ens_mean_f000-f120_6h.grib2   ensemble mean (6-hourly, 0–120h)
    aifs_ens_spread_f000-f120_6h.grib2  ensemble spread
```

In [ ]:
from ecmwf.opendata import Client
from pathlib import Path

BASE_DIR = Path("data")
BASE_DIR.mkdir(parents=True, exist_ok=True)
print("Ready. BASE_DIR:", BASE_DIR.resolve())

In [ ]:
client = Client(source="ecmwf", model="aifs-ens")
latest_run  = client.latest()
latest_time = latest_run.hour

run_label  = f"{latest_run.strftime('%Y-%m-%d')}_{latest_time:02d}Z"
output_dir = BASE_DIR / run_label
output_dir.mkdir(parents=True, exist_ok=True)

print("Latest run  :", latest_run)
print("Cycle (UTC) :", latest_time)
print("Output dir  :", output_dir.resolve())

In [ ]:
# ── AIFS-ENS mean + spread — 6-hourly, 0–120h ────────────────────────────────
# em/es provides only 3 params; for the full AIFS surface set use the cf cells below.
steps_6h = list(range(0, 121, 6))

AIFS_EM_SFC = {}
AIFS_EM_SFC["thermo"]   = ["2t"]
AIFS_EM_SFC["pressure"] = ["msl"]
AIFS_EM_SFC["wind"]     = ["10si"]   # scalar speed only — no u/v in em/es

for type_, label in [("em", "aifs_ens_mean"), ("es", "aifs_ens_spread")]:
    client.retrieve(
        time=latest_time,
        step=steps_6h,
        type=type_,
        param=[p for grp in AIFS_EM_SFC.values() for p in grp],
        target=str(output_dir / f"{label}_f000-f120_6h.grib2"),
    )
    print(f"Saved: {label}_f000-f120_6h.grib2")

In [ ]:
# ── NEW: AIFS-ENS control forecast surface — 6-hourly, 0–120h ────────────────
# cf gives the full AIFS surface parameter set (same as AIFS-single)
AIFS_CF_SFC = {}
AIFS_CF_SFC["thermo"]   = ["2t", "2d", "skt", "sst"]
AIFS_CF_SFC["wind"]     = ["10u", "10v", "10si", "100u", "100v"]
AIFS_CF_SFC["pressure"] = ["msl"]
AIFS_CF_SFC["moisture"] = ["tp", "cp", "tcc", "lcc", "mcc", "hcc", "tcw"]
AIFS_CF_SFC["energy"]   = ["ssrd", "strd"]

client.retrieve(
    time=latest_time,
    step=steps_6h,
    type="cf",
    param=[p for grp in AIFS_CF_SFC.values() for p in grp],
    target=str(output_dir / "aifs_ens_cf_sfc_f000-f120_6h.grib2"),
)
print("Saved: aifs_ens_cf_sfc_f000-f120_6h.grib2")
print(f"  Groups: { {k: len(v) for k, v in AIFS_CF_SFC.items()} }")

In [ ]:
# ── NEW: AIFS-ENS control forecast upper-air — 12-hourly, 0–120h ─────────────
# AIFS does not output r, vo, d — only t, gh, u, v, w, q available
AIFS_CF_PL = {}
AIFS_CF_PL["thermo"]   = ["t", "gh"]
AIFS_CF_PL["wind"]     = ["u", "v", "w"]
AIFS_CF_PL["moisture"] = ["q"]

client.retrieve(
    time=latest_time,
    step=list(range(0, 121, 12)),
    type="cf",
    param=[p for grp in AIFS_CF_PL.values() for p in grp],
    levtype="pl", levelist=[1000, 925, 850, 700, 600, 500, 400, 300, 250, 200],
    target=str(output_dir / "aifs_ens_cf_pl_f000-f120_12h.grib2"),
)
print("Saved: aifs_ens_cf_pl_f000-f120_12h.grib2")